# Laboratorio de regresión - 4

|                |   |
:----------------|---|
| **Nombre**     |  Jorge Oviedo Magaña |
| **Fecha**      |  08/02/2026 |
| **Expediente** |  757048|

## Modelos penalizados

Hasta ahora la función de costo que usamos para decidir qué tan bueno es nuestro modelo al momento de ajustar es:

$$ \text{RSS} = \sum_{i=1}^n e_i^2 = \sum_{i=1}^n (y_i - \hat{y_i})^2 $$

Dado que los errores obtenidos son una combinación de sesgo y varianza, puede ser que se sesgue un parámetro para minimizar el error. Esto significa que el modelo puede decidir que la salida no sea una combinación de los factores, sino una fuerte predilección sobre uno de los factores solamente. 

E.g. se quiere ajustar un modelo

$$ \hat{z} = \hat{\beta_0} + \hat{\beta_1} x + \hat{\beta_2} y $$

Se ajusta el modelo y se decide que la mejor decisión es $\hat{\beta_1} = 10000$ y $\hat{\beta_2}=50$. Considera limitaciones de problemas reales:
- Quizás los parámetros son ajustes de maquinaria que se deben realizar para conseguir el mejor producto posible, y que $10000$ sea imposible de asignar.
- Quizás los datos actuales están sesgados y sólo hacen parecer que uno de los factores importa más que el otro.

Una de las formas en las que se puede mitigar este problema es penalizando a los parámetros del modelo, cambiando la función de costo:

$$ \text{RSS}_{L2} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p \hat{\beta_j}^2 $$

El *L2* significa que se está agregando una penalización de segundo orden. Lo que hace esta penalización es que los factores ahora sólo tendrán permitido crecer si hay una reducción al menos proporcional en el error (sacrificamos sesgo, pero reducimos la varianza).

Asimismo, existe la penalización *L1*

$$ \text{RSS}_{L1} = \sum_{i=1}^n e_i^2  + \lambda \sum_{j=1}^p |\hat{\beta_j}| $$

A las penalizaciones *L2* y *L1* se les conoce también como Ridge y Lasso, respectivamente.

Para realizar una regresión con penalización de Ridge o de Lasso usamos el objeto `Ridge(alpha=?)` o `Lasso(alpha=?)` en lugar de `LinearRegression()` de `sklearn`.

Utiliza el dataset de publicidad (Advertising.csv) y realiza 3 regresiones múltiples:

$$ \text{sales} = \beta_0 + \beta_1 (\text{TV}) + \beta_2 (\text{radio}) + \beta_3 (\text{newspaper}) + \epsilon $$

1. Sin penalización
2. Con penalización L2
3. Con penalización L1

¿Qué puedes observar al ajustar los valores de `alpha`? 

Compara los resultados de los coeficientes utilizando valores diferentes de $\alpha$ y los $R^2$ resultantes.



In [5]:
import pandas as pd

Ad = pd.read_csv('Advertising.csv')

Ad.head()

,Unnamed: 0,TV,radio,newspaper,sales
0,1,230.1,37.8,69.2,22.1
1,2,44.5,39.3,45.1,10.4
2,3,17.2,45.9,69.3,9.3
3,4,151.5,41.3,58.5,18.5
4,5,180.8,10.8,58.4,12.9


In [ ]:
# regresion multiple sin penalización [sales = b0 + b1*TV + b2*radio + b3*newspaper + e]
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
X = Ad.drop('sales', axis=1)
y = Ad['sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error: {mse}')



Mean Squared Error: 3.1990044685889045


In [7]:
# Ridge Regression
from sklearn.linear_model import Ridge, Lasso

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
ridge_y_pred = ridge_model.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_y_pred)
print(f'Ridge Mean Squared Error: {ridge_mse}')


Ridge Mean Squared Error: 3.199001864104396


In [8]:
# Lasso Regression
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train, y_train)
lasso_y_pred = lasso_model.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_y_pred)
print(f'Lasso Mean Squared Error: {lasso_mse}')

Lasso Mean Squared Error: 3.193700795905665


In [ ]:
# Comparación de coeficientes
print(f'Linear Regression Coefficients: {model.coef_}')

alphas = [0.1, 1.0, 10.0]
for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)
    ridge_model.fit(X_train, y_train)
    ridge_y_pred = ridge_model.predict(X_test)
    ridge_mse = mean_squared_error(y_test, ridge_y_pred)
    print(f'Ridge (alpha={alpha}) Mean Squared Error: {ridge_mse}')
    print(f'Ridge (alpha={alpha}) Coefficients: {ridge_model.coef_}')
    
    lasso_model = Lasso(alpha=alpha)
    lasso_model.fit(X_train, y_train)
    lasso_y_pred = lasso_model.predict(X_test)
    lasso_mse = mean_squared_error(y_test, lasso_y_pred)
    print(f'Lasso (alpha={alpha}) Mean Squared Error: {lasso_mse}')
    print(f'Lasso (alpha={alpha}) Coefficients: {lasso_model.coef_}')



Linear Regression Coefficients: [0.00064359 0.04471835 0.18925118 0.00304577]
Ridge (alpha=0.1) Mean Squared Error: 3.199004207369359
Ridge (alpha=0.1) Coefficients: [0.00064359 0.04471835 0.18925055 0.00304594]
Lasso (alpha=0.1) Mean Squared Error: 3.193700795905665
Lasso (alpha=0.1) Coefficients: [0.00059941 0.04470908 0.18886564 0.00289154]
Ridge (alpha=1.0) Mean Squared Error: 3.199001864104396
Ridge (alpha=1.0) Coefficients: [0.00064356 0.04471836 0.18924481 0.0030475 ]
Lasso (alpha=1.0) Mean Squared Error: 3.151352414297348
Lasso (alpha=1.0) Coefficients: [0.00020037 0.04462722 0.18536764 0.00151066]
Ridge (alpha=10.0) Mean Squared Error: 3.198979194244958
Ridge (alpha=10.0) Coefficients: [0.00064327 0.04471848 0.1891875  0.00306313]
Lasso (alpha=10.0) Mean Squared Error: 3.4573980270909197
Lasso (alpha=10.0) Coefficients: [-0.          0.04374718  0.14506012  0.        ]


 En los resultados podemos observar que a medida que aumentamos el valor de alpha, los coeficientes de Ridge se vuelven más pequeños, pero no llegan a ser exactamente cero, en cambio con Lasso algunos coeficientes se vuelven exactamente cero, lo que indica que esas características han sido completamente eliminadas del modelo. Esto es especialmente útil para la selección de características.